# Train Siamese LSTM Truyen Thong ? Triplet Cosine

Notebook nay dung pipeline `data_ready` giong `train-siamese-bilstm-cosine.ipynb`, nhung encoder la **LSTM 1 chieu (traditional)** va toi uu voi **triplet Cosine loss** theo baseline paper.


In [1]:
import os
import re
import json
import random
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

def simple_tokenize(text: str):
    text = str(text).lower().strip()
    return re.findall(r"\w+", text, flags=re.UNICODE)


def load_artifacts(artifact_dir):
    artifact_dir = Path(artifact_dir)
    with open(artifact_dir / 'tokenizer_vocab.json', 'r', encoding='utf-8') as f:
        vocab = json.load(f)

    ckpt = torch.load(artifact_dir / 'embedding.pt', map_location='cpu')
    stoi = vocab['stoi']
    embedding_weight = ckpt['embedding_weight']
    pad_idx = int(ckpt['pad_idx'])
    return stoi, embedding_weight, pad_idx


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    torch.backends.cudnn.benchmark = True


Device: cuda
GPU: Tesla T4


In [2]:
def detect_data_dir():
    candidates = [
        Path('/kaggle/input/datasets/hngphtrn/legals/data/data_ready'),
        Path('/kaggle/input/datasets/hngphtrn/legals/data_ready'),
        Path('/kaggle/input/vnlegal-rag/data/data_ready'),
        Path('/kaggle/input/vnlegal-rag/data_ready'),
        Path('/kaggle/working/vnlegal-rag/data/data_ready'),
        Path('/kaggle/working/legals/data/data_ready'),
        Path('/kaggle/working/data/data_ready'),
        Path('data/data_ready'),
        Path('../data/data_ready'),
    ]
    for p in candidates:
        if p.exists() and (p / 'qa_train.csv').exists() and (p / 'corpus_train.csv').exists():
            return p
    raise FileNotFoundError('Khong tim thay data_ready. Hay kiem tra duong dan dataset tren Kaggle.')


DATA_DIR = detect_data_dir()
ARTIFACT_DIR = Path('/kaggle/working/siamese_lstm_traditional_cosine_artifacts' if Path('/kaggle').exists() else 'artifacts/siamese_lstm_traditional_cosine')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print('DATA_DIR =', DATA_DIR)
print('ARTIFACT_DIR =', ARTIFACT_DIR)


DATA_DIR = /kaggle/input/datasets/hngphtrn/legals/data_ready
ARTIFACT_DIR = /kaggle/working/siamese_lstm_traditional_cosine_artifacts


In [3]:
qa_train = pd.read_csv(DATA_DIR / 'qa_train.csv', sep='	')
qa_val = pd.read_csv(DATA_DIR / 'qa_val.csv', sep='	')
qa_test = pd.read_csv(DATA_DIR / 'qa_test.csv', sep='	')
corpus_train = pd.read_csv(DATA_DIR / 'corpus_train.csv', sep='	')
corpus_val = pd.read_csv(DATA_DIR / 'corpus_val.csv', sep='	')
corpus_test = pd.read_csv(DATA_DIR / 'corpus_test.csv', sep='	')

print('qa_train:', qa_train.shape, '| corpus_train:', corpus_train.shape)
print('qa_val:  ', qa_val.shape, '| corpus_val:  ', corpus_val.shape)
print('qa_test: ', qa_test.shape, '| corpus_test: ', corpus_test.shape)


def pick_first_existing_col(df, candidates, df_name):
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"Khong tim thay cot nao trong {candidates} cho {df_name}. Cot hien co: {list(df.columns)}")


QA_TEXT_COL = pick_first_existing_col(qa_train, ['question', 'query', 'text', 'question_text'], 'qa_train')
CORPUS_TEXT_COL = pick_first_existing_col(
    corpus_train,
    ['article_chunk', 'article_content', 'context', 'text', 'content', 'chunk_text', 'passage_text'],
    'corpus_train',
)
print('QA_TEXT_COL =', QA_TEXT_COL)
print('CORPUS_TEXT_COL =', CORPUS_TEXT_COL)


qa_train: (23311, 14) | corpus_train: (7771, 6)
qa_val:   (2841, 14) | corpus_val:   (947, 6)
qa_test:  (2991, 14) | corpus_test:  (997, 6)
QA_TEXT_COL = question
CORPUS_TEXT_COL = article_content


In [4]:
try:
    from underthesea import word_tokenize
    USE_UNDERTHESEA = True
except Exception:
    USE_UNDERTHESEA = False
    print('underthesea chua co. Chay cell sau de cai dat neu can.')


def normalize_text(text: str) -> str:
    text = str(text).lower().strip()
    text = re.sub(r'\s+', ' ', text)
    return text


def tokenize(text: str):
    text = normalize_text(text)
    if USE_UNDERTHESEA:
        return word_tokenize(text, format='text').split()
    # Fallback robust on Kaggle/Python 3.12 when underthesea is unavailable
    return re.findall(r"\w+", text, flags=re.UNICODE)


underthesea chua co. Chay cell sau de cai dat neu can.


In [5]:
# Underthesea co the loi build tren Python 3.12 (Kaggle) vi phu thuoc native.
# Neu ban dung Python 3.10/3.11, co the thu:
# !pip -q install underthesea==6.8.4
#
# Neu van loi, giu fallback tokenizer (regex) va bo qua cai dat underthesea.
# Fallback da du de train baseline va so sanh metric.


In [6]:
PAD = '<PAD>'
UNK = '<UNK>'
MAX_Q_LEN = 64
MAX_D_LEN = 256
SHARED_EMBED_DIR = Path('/kaggle/input/datasets/hngphtrn/shared-embedding-artifacts') if Path('/kaggle/input/datasets/hngphtrn/shared-embedding-artifacts').exists() else Path('shared_embedding_artifacts')

stoi, embedding_weight, pad_idx = load_artifacts(SHARED_EMBED_DIR)
itos = {i: w for w, i in stoi.items()}
MAX_VOCAB = len(stoi)

vocab_meta = json.loads((SHARED_EMBED_DIR / 'tokenizer_vocab.json').read_text(encoding='utf-8'))
build_meta = vocab_meta.get('build_metadata', {})
if not build_meta.get('corpus_path'):
    print('WARNING: shared vocab may not include corpus_train.csv. Rebuild shared embedding artifacts first.')


def encode_text(text, max_len):
    toks = simple_tokenize(text)
    ids = [stoi.get(t, stoi[UNK]) for t in toks[:max_len]]
    if len(ids) < max_len:
        ids += [stoi[PAD]] * (max_len - len(ids))
    return ids

print('Loaded shared vocab size:', len(stoi), '| embedding shape:', tuple(embedding_weight.shape))
print('Shared vocab built from:', build_meta.get('qa_path', 'unknown'), 'and', build_meta.get('corpus_path', 'unknown'))


Loaded shared vocab size: 6227 | embedding shape: (6227, 200)
Shared vocab built from: data\data_ready\qa_train.csv and data\data_ready\corpus_train.csv


In [7]:
# ── corpus_full: dùng làm search space khi validate (thực tế hơn)
corpus_full = pd.concat([corpus_train, corpus_val, corpus_test], ignore_index=True)
corpus_full_map = corpus_full.set_index('passage_id').to_dict(orient='index')

corpus_train_map = corpus_train.set_index('passage_id').to_dict(orient='index')
corpus_val_map = corpus_val.set_index('passage_id').to_dict(orient='index')
corpus_test_map = corpus_test.set_index('passage_id').to_dict(orient='index')

domain_to_passages = defaultdict(list)
if 'macro_domain' in corpus_train.columns:
    for _, row in corpus_train.iterrows():
        domain_to_passages[str(row.get('macro_domain', 'unknown'))].append(row['passage_id'])


## Hard Negative Mining via TF-IDF

Trước khi khởi tạo dataset, build TF-IDF index trên `corpus_train` để
tìm hard negatives (đoạn khó phân biệt nhất với câu hỏi) thay vì chọn ngẫu nhiên.

In [8]:
# ── TF-IDF Hard Negative Mining ────────────────────────────────────────────
# Thay vì chọn negative ngẫu nhiên trong cùng domain, ta dùng TF-IDF
# để tìm những đoạn văn "trông giống" câu hỏi nhất (nhưng không phải đáp án)
# → model phải học phân biệt những cặp khó hơn.
#
# Chiến lược "offline periodic refresh":
#   - Trước epoch 1 và mỗi HARD_NEG_REFRESH_EVERY epoch, recompute
#     top-k TF-IDF candidates cho toàn bộ query train.
#   - Trong __getitem__, sample 1 negative từ top-k candidates (bỏ positive).
# ──────────────────────────────────────────────────────────────────────────

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize as sk_normalize
import numpy as _np_hn

HARD_NEG_TOPK   = 30   # số TF-IDF candidates giữ lại mỗi query
HARD_NEG_REFRESH_EVERY = 3   # refresh lại sau bao nhiêu epoch (0 = chỉ 1 lần)

# Build TF-IDF trên corpus_train (chỉ cần 1 lần)
_corpus_texts_hn = corpus_train[CORPUS_TEXT_COL].astype(str).tolist()
_corpus_pids_hn  = corpus_train["passage_id"].tolist()

print("Building TF-IDF index for hard negative mining ...")
_tfidf_hn = TfidfVectorizer(
    max_features=60_000,
    ngram_range=(1, 2),
    min_df=2,
    analyzer="word",
    tokenizer=simple_tokenize,   # dùng cùng tokenizer với Siamese
    token_pattern=None,
)
_tfidf_corpus_mat = _tfidf_hn.fit_transform(_corpus_texts_hn)   # (N_corpus, V)
_tfidf_corpus_mat = sk_normalize(_tfidf_corpus_mat, norm="l2")  # cosine-ready
print(f"TF-IDF index: {_tfidf_corpus_mat.shape[0]} docs x {_tfidf_corpus_mat.shape[1]} terms")


def build_hard_neg_lookup(qa_df, topk=HARD_NEG_TOPK):
    """
    Trả về dict: query_integer_index → list[passage_id] (hard negative candidates).
    Positive passage_id bị loại khỏi danh sách.
    """
    q_texts = qa_df[QA_TEXT_COL].astype(str).tolist()
    pos_pids = qa_df["passage_id"].tolist()

    # Encode all queries at once (nhanh hơn từng cái)
    q_mat = _tfidf_hn.transform(q_texts)           # (N_q, V)
    q_mat = sk_normalize(q_mat, norm="l2")

    scores = (q_mat @ _tfidf_corpus_mat.T).toarray()  # (N_q, N_corpus)

    lookup = {}
    for i, (row_scores, pos_pid) in enumerate(zip(scores, pos_pids)):
        # Lấy top-(topk+1) để có dư sau khi lọc positive
        top_idx = _np_hn.argpartition(-row_scores, kth=min(topk + 1, len(row_scores) - 1))[: topk + 5]
        top_idx = top_idx[_np_hn.argsort(-row_scores[top_idx])]
        cands = [_corpus_pids_hn[j] for j in top_idx if _corpus_pids_hn[j] != pos_pid][:topk]
        lookup[i] = cands if cands else None   # None → fallback random
    return lookup


# Tính lần đầu
hard_neg_lookup = build_hard_neg_lookup(qa_train, topk=HARD_NEG_TOPK)
_nonzero = sum(1 for v in hard_neg_lookup.values() if v)
print(f"Hard neg lookup built: {_nonzero}/{len(hard_neg_lookup)} queries có candidates")


Building TF-IDF index for hard negative mining ...
TF-IDF index: 7771 docs x 60000 terms
Hard neg lookup built: 23311/23311 queries có candidates


In [9]:
class HardNegTripletDataset(Dataset):
    """
    Triplet dataset với hard negative mining qua TF-IDF.

    hard_neg_lookup: dict[int, list[str] | None]
        Mapping từ query index → list passage_id là hard negative candidates.
        Nếu None hoặc rỗng, fallback về random in-domain (giống trước).
    fallback_domain_map: dict  (domain_to_passages cũ)
    """
    def __init__(self, qa_df, corpus_map, domain_to_passages, hard_neg_lookup=None):
        self.qa_df           = qa_df.reset_index(drop=True)
        self.corpus_map      = corpus_map
        self.domain_map      = domain_to_passages
        self.hard_neg_lookup = hard_neg_lookup or {}
        self._corpus_keys    = list(corpus_map.keys())

    def __len__(self):
        return len(self.qa_df)

    def refresh_hard_negs(self, new_lookup):
        """Gọi đầu mỗi epoch để cập nhật lookup (periodic refresh)."""
        self.hard_neg_lookup = new_lookup

    def _sample_negative(self, idx, pos_pid, macro_domain):
        # ── Ưu tiên hard negative từ TF-IDF ──
        cands = self.hard_neg_lookup.get(idx)
        if cands:
            # Lọc lại pos_pid (phòng trường hợp lọt qua)
            valid = [p for p in cands if p in self.corpus_map and p != pos_pid]
            if valid:
                return random.choice(valid)

        # ── Fallback: random in-domain (giống behaviour cũ) ──
        if self.domain_map and macro_domain in self.domain_map:
            domain_cands = [p for p in self.domain_map[macro_domain]
                            if p != pos_pid and p in self.corpus_map]
            if domain_cands:
                return random.choice(domain_cands)

        # ── Fallback cuối: random toàn corpus ──
        neg = random.choice(self._corpus_keys)
        while neg == pos_pid:
            neg = random.choice(self._corpus_keys)
        return neg

    def __getitem__(self, idx):
        row     = self.qa_df.iloc[idx]
        q       = str(row[QA_TEXT_COL])
        pos_pid = row["passage_id"]
        macro   = str(row.get("macro_domain", "unknown"))

        pos_doc = self.corpus_map[pos_pid]
        neg_pid = self._sample_negative(idx, pos_pid, macro)
        neg_doc = self.corpus_map[neg_pid]

        return {
            "anchor":   torch.tensor(encode_text(q, MAX_Q_LEN), dtype=torch.long),
            "positive": torch.tensor(encode_text(str(pos_doc[CORPUS_TEXT_COL]), MAX_D_LEN), dtype=torch.long),
            "negative": torch.tensor(encode_text(str(neg_doc[CORPUS_TEXT_COL]), MAX_D_LEN), dtype=torch.long),
        }


# Giữ lại class cũ dưới tên alias để tương thích nếu cần
TripletQADataset = HardNegTripletDataset


In [10]:
# hard_neg_lookup được build ở cell trước (TF-IDF setup).
# Sẽ được refresh định kỳ trong training loop.
train_ds = HardNegTripletDataset(
    qa_train,
    corpus_train_map,
    domain_to_passages,
    hard_neg_lookup=hard_neg_lookup,
)

NUM_WORKERS = 0
train_loader = DataLoader(
    train_ds,
    batch_size=32,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=(NUM_WORKERS > 0),
)
print("Train triplets:", len(train_ds))
_hard_frac = sum(1 for v in hard_neg_lookup.values() if v) / max(1, len(hard_neg_lookup))
print(f"  Hard neg coverage: {_hard_frac*100:.1f}% queries có TF-IDF candidates")


Train triplets: 23311
  Hard neg coverage: 100.0% queries có TF-IDF candidates


In [11]:
class LSTMEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=200, hidden_size=698, num_layers=1, dropout=0.3, pad_idx=0, embedding_weight=None):
        super().__init__()
        if embedding_weight is None:
            self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        else:
            self.embedding = nn.Embedding.from_pretrained(
                embedding_weight,
                freeze=False,
                padding_idx=pad_idx,
            )
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=False,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.pad_idx = pad_idx

    def forward(self, x):
        mask = (x != self.pad_idx).float().unsqueeze(-1)
        emb = self.embedding(x)
        out, _ = self.lstm(emb)
        summed = (out * mask).sum(dim=1)
        denom = mask.sum(dim=1).clamp(min=1e-6)
        pooled = summed / denom
        return F.normalize(pooled, p=2, dim=1)


class SiameseLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=200, hidden_size=698, num_layers=1, dropout=0.3, pad_idx=0, embedding_weight=None):
        super().__init__()
        self.encoder = LSTMEncoder(vocab_size, embed_dim, hidden_size, num_layers, dropout, pad_idx, embedding_weight=embedding_weight)

    def encode(self, x):
        return self.encoder(x)

    def forward(self, a, p, n):
        return self.encode(a), self.encode(p), self.encode(n)


In [12]:
EMBED_DIM = int(embedding_weight.shape[1])
HIDDEN_SIZE = 698
NUM_LAYERS = 1
ENCODER_DROPOUT = 0.3

model = SiameseLSTM(
    vocab_size=len(stoi),
    embed_dim=EMBED_DIM,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    dropout=ENCODER_DROPOUT,
    pad_idx=stoi['<PAD>'],
    embedding_weight=embedding_weight,
).to(device)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable params: {trainable_params:,}')

optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=1)
scaler = GradScaler('cuda', enabled=torch.cuda.is_available())
margin = 0.3


def triplet_loss_cosine(a, p, n, margin=0.3):
    sim_ap = F.cosine_similarity(a, p)
    sim_an = F.cosine_similarity(a, n)
    return F.relu(margin - sim_ap + sim_an).mean()


Trainable params: 3,758,200


In [13]:
def train_one_epoch(model, loader):
    model.train()
    total = 0.0
    for batch in tqdm(loader, leave=False):
        a = batch['anchor'].to(device)
        p = batch['positive'].to(device)
        n = batch['negative'].to(device)

        optimizer.zero_grad(set_to_none=True)
        with autocast('cuda', enabled=torch.cuda.is_available()):
            za, zp, zn = model(a, p, n)
            loss = triplet_loss_cosine(za, zp, zn, margin=margin)

        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        total += float(loss.item())
    return total / max(1, len(loader))


In [14]:
@torch.no_grad()
def encode_passage_matrix(model, corpus_df, batch_size=64):
    model.eval()
    pids = corpus_df['passage_id'].tolist()
    vecs = []
    for i in range(0, len(pids), batch_size):
        chunk = pids[i:i + batch_size]
        texts = corpus_df.iloc[i:i + batch_size][CORPUS_TEXT_COL].astype(str).tolist()
        ids = torch.tensor([encode_text(t, MAX_D_LEN) for t in texts], dtype=torch.long, device=device)
        z = model.encode(ids)
        vecs.append(z.cpu())
    return pids, torch.cat(vecs, dim=0)


@torch.no_grad()
def evaluate_retrieval(model, qa_df, corpus_df, topk=(1, 3, 5), max_queries=1500):
    model.eval()
    pids, pmat = encode_passage_matrix(model, corpus_df, batch_size=64)
    pid_to_idx = {p: i for i, p in enumerate(pids)}

    qa_eval = qa_df if (max_queries is None or len(qa_df) <= max_queries) else qa_df.sample(max_queries, random_state=42)

    hits = {k: 0 for k in topk}
    rr_sum = 0.0

    for _, row in tqdm(qa_eval.iterrows(), total=len(qa_eval), leave=False):
        q_ids = torch.tensor([encode_text(str(row[QA_TEXT_COL]), MAX_Q_LEN)], dtype=torch.long, device=device)
        q_vec = model.encode(q_ids).cpu()

        scores = torch.matmul(q_vec, pmat.T).squeeze(0)
        order = torch.argsort(scores, descending=True)

        true_pid = row['passage_id']
        true_idx = pid_to_idx.get(true_pid, None)
        if true_idx is None:
            continue

        rank_pos = (order == true_idx).nonzero(as_tuple=True)[0]
        if len(rank_pos) == 0:
            continue
        rank = int(rank_pos.item()) + 1
        rr_sum += 1.0 / rank

        for k in topk:
            if rank <= k:
                hits[k] += 1

    n = len(qa_eval)
    metrics = {f'Recall@{k}': hits[k] / max(1, n) for k in topk}
    metrics['MRR'] = rr_sum / max(1, n)
    return metrics


In [15]:
EPOCHS               = 20
PATIENCE             = 5
MIN_DELTA            = 1e-4
EMBED_FREEZE_EPOCHS  = int(globals().get('EMBED_FREEZE_EPOCHS', 0))
embedding_params     = list(globals().get("embedding_params", model.encoder.embedding.parameters()))
best_score  = -1.0
best_epoch  = 0
wait        = 0
best_path   = ARTIFACT_DIR / "siamese_lstm_traditional_cosine_best.pt"
history     = []

for epoch in range(1, EPOCHS + 1):
    # ── Periodic hard negative refresh ──────────────────────────────────────
    # Epoch 1: đã build trước. Refresh mỗi HARD_NEG_REFRESH_EVERY epoch.
    if HARD_NEG_REFRESH_EVERY > 0 and epoch > 1 and (epoch - 1) % HARD_NEG_REFRESH_EVERY == 0:
        print(f"  [HardNeg] Refreshing TF-IDF hard negatives (epoch {epoch}) ...")
        hard_neg_lookup = build_hard_neg_lookup(qa_train, topk=HARD_NEG_TOPK)
        train_ds.refresh_hard_negs(hard_neg_lookup)
        print(f"  [HardNeg] Done.")
    # ────────────────────────────────────────────────────────────────────────

    freeze_embedding = epoch <= EMBED_FREEZE_EPOCHS
    for p in embedding_params:
        p.requires_grad = (not freeze_embedding)

    tr_loss    = train_one_epoch(model, train_loader)
    val_metrics = evaluate_retrieval(model, qa_val, corpus_full, topk=(1, 3, 5), max_queries=1500)
    val_score  = val_metrics["MRR"]
    scheduler.step(val_score)

    row = {"epoch": epoch, "train_loss": tr_loss, "freeze_embedding": freeze_embedding, **val_metrics}
    history.append(row)
    phase = "frozen" if freeze_embedding else "finetune"
    print(
        f"Epoch {epoch:02d} [{phase}] | loss={tr_loss:.4f}"
        f" | MRR={val_metrics['MRR']:.4f}"
        f" | R@1={val_metrics['Recall@1']:.4f}"
        f" | R@5={val_metrics['Recall@5']:.4f}"
    )

    if val_score > best_score + MIN_DELTA:
        best_score  = val_score
        best_epoch  = epoch
        wait        = 0
        torch.save(model.state_dict(), best_path)
        print(f"  -> Saved best checkpoint at epoch {epoch:02d} (MRR={best_score:.4f})")
    else:
        wait += 1
        if wait >= PATIENCE:
            print(f"Early stopping at epoch {epoch:02d}. Best epoch: {best_epoch:02d} (MRR={best_score:.4f})")
            break

print(f"Best val MRR: {best_score:.4f} (epoch {best_epoch:02d})")
print(f"Best checkpoint: {best_path}")


  0%|          | 0/729 [00:00<?, ?it/s]

  0%|          | 0/1500 [00:00<?, ?it/s]

Epoch 01 [finetune] | loss=0.1464 | MRR=0.2831 | R@1=0.1860 | R@5=0.3927
  -> Saved best checkpoint at epoch 01 (MRR=0.2831)


  0%|          | 0/729 [00:00<?, ?it/s]

  0%|          | 0/1500 [00:00<?, ?it/s]

...

  0%|          | 0/1500 [00:00<?, ?it/s]

Epoch 19 [finetune] | loss=0.0324 | MRR=0.4180 | R@1=0.2940 | R@5=0.5560
  -> Saved best checkpoint at epoch 19 (MRR=0.4180)


  0%|          | 0/729 [00:00<?, ?it/s]

  0%|          | 0/1500 [00:00<?, ?it/s]

Epoch 20 [finetune] | loss=0.0327 | MRR=0.4207 | R@1=0.3027 | R@5=0.5673
  -> Saved best checkpoint at epoch 20 (MRR=0.4207)
Best val MRR: 0.4207 (epoch 20)
Best checkpoint: /kaggle/working/siamese_lstm_traditional_cosine_artifacts/siamese_lstm_traditional_cosine_best.pt


In [16]:
model.load_state_dict(torch.load(best_path, map_location=device))
test_metrics = evaluate_retrieval(model, qa_test, corpus_test, topk=(1, 3, 5, 10), max_queries=None)
print('Test metrics:', test_metrics)


  0%|          | 0/2991 [00:00<?, ?it/s]

Test metrics: {'Recall@1': 0.5025075225677031, 'Recall@3': 0.6552992310264125, 'Recall@5': 0.720160481444333, 'Recall@10': 0.792711467736543, 'MRR': 0.6013913518124816}


In [17]:
with open(ARTIFACT_DIR / 'tokenizer_vocab.json', 'w', encoding='utf-8') as f:
    json.dump({'stoi': stoi, 'itos': {str(k): v for k, v in itos.items()}}, f, ensure_ascii=False)

with open(ARTIFACT_DIR / 'siamese_meta.json', 'w', encoding='utf-8') as f:
    json.dump(
        {
            'distance_metric': 'cosine',
            'encoder': 'lstm_traditional_cosine',
            'max_q_len': MAX_Q_LEN,
            'max_d_len': MAX_D_LEN,
            'margin': margin,
            'embed_dim': EMBED_DIM,
            'hidden_size': HIDDEN_SIZE,
            'num_layers': NUM_LAYERS,
            'dropout': ENCODER_DROPOUT,
            'trainable_params': trainable_params,
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

pd.DataFrame(history).to_csv(ARTIFACT_DIR / 'train_history.csv', index=False)
print('Saved artifacts at:', ARTIFACT_DIR)


Saved artifacts at: /kaggle/working/siamese_lstm_traditional_cosine_artifacts
